# PLaMo 2 1B: Equal-24,883,200-Token peS2o Continued Pretraining

This notebook trains one data variant per run: `raw`, `minhashlsh`, or `lshbloom`. Every run uses exactly 24,883,200 PLaMo tokenizer tokens, so the training-data deduplication method is the only experimental variable. This common budget is the largest multiple of 2,048 that fits the smallest variant after all three complete files are counted with the pinned PLaMo tokenizer.

PLaMo 2 uses custom Mamba kernels with strict dependency requirements. Before connecting, select **Runtime > Change runtime type**, choose an **A100 GPU**, and choose the **2025.07 past runtime (Python 3.11)**. The setup cell installs all pinned dependencies and restarts the runtime once. After reconnection, run all cells again.

Add `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `WANDB_API_KEY` to Colab Secrets. Temporary AWS credentials also require `AWS_SESSION_TOKEN`. Use a fresh runtime for each variant.


In [ ]:
# Install the exact PLaMo-compatible runtime dependencies.
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path


REQUIRED = {
    "torch_version": "2.5.1",
    "transformers_version": "4.57.1",
    "accelerate_version": "1.10.1",
    "numpy_version": "2.0.2",
    "numba_version": "0.60.0",
    "mamba_ssm_version": "2.2.4",
    "causal_conv1d_version": "1.4.0",
}
SETUP_MARKER = Path("/content/.plamo2_dependency_setup_v2")
EXACT_DISTRIBUTIONS = {
    "torch": REQUIRED["torch_version"],
    "transformers": REQUIRED["transformers_version"],
    "accelerate": REQUIRED["accelerate_version"],
    "numpy": REQUIRED["numpy_version"],
    "numba": REQUIRED["numba_version"],
    "mamba-ssm": REQUIRED["mamba_ssm_version"],
    "causal-conv1d": REQUIRED["causal_conv1d_version"],
}

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Select Colab past runtime 2025.07, which provides Python 3.11, "
        f"then reconnect. Current Python: {sys.version.split()[0]}"
    )

def installed_version(distribution):
    try:
        return importlib.metadata.version(distribution).split("+")[0]
    except importlib.metadata.PackageNotFoundError:
        return None


def environment_matches():
    return all(
        installed_version(distribution) == expected
        for distribution, expected in EXACT_DISTRIBUTIONS.items()
    )


if not SETUP_MARKER.exists() or not environment_matches():
    if installed_version("torch") != REQUIRED["torch_version"]:
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
            check=False,
        )
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torch=={REQUIRED['torch_version']}",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ])

    # Reinstall NumPy even when its metadata matches. This repairs a runtime that
    # previously loaded a different NumPy version before pip changed the files.
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        "--no-deps",
        f"numpy=={REQUIRED['numpy_version']}",
    ])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"transformers=={REQUIRED['transformers_version']}",
        f"accelerate=={REQUIRED['accelerate_version']}",
        f"numba=={REQUIRED['numba_version']}",
        "wandb==0.21.1",
        "boto3>=1.35,<2",
        "packaging>=24,<26",
        "ninja>=1.11,<2",
        "wheel",
    ])

    # Limit extension compilation memory on the Colab VM.
    os.environ["MAX_JOBS"] = "2"
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        "--no-build-isolation",
        "--no-deps",
        f"causal-conv1d=={REQUIRED['causal_conv1d_version']}",
        f"mamba-ssm=={REQUIRED['mamba_ssm_version']}",
    ])
    SETUP_MARKER.write_text("installed", encoding="utf-8")
    print("All PLaMo dependencies are installed. Colab will restart once.")
    os.kill(os.getpid(), 9)

# Import the same modules used by training before any data or model download.
try:
    import numpy as setup_numpy
    import numpy.rec
    import causal_conv1d as setup_causal_conv1d
    import mamba_ssm as setup_mamba_ssm
    from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
except Exception as error:
    SETUP_MARKER.unlink(missing_ok=True)
    raise RuntimeError(
        "The dependency self-check failed. Disconnect and delete this Colab "
        "runtime, select 2025.07 again, and rerun the notebook."
    ) from error

if setup_numpy.__version__ != REQUIRED["numpy_version"]:
    raise RuntimeError(
        f"Expected NumPy {REQUIRED['numpy_version']}, received {setup_numpy.__version__}"
    )
print("PLaMo dependency self-check passed.")


In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import os
import tempfile
from pathlib import Path
from typing import Iterator

import numpy as np


def _iter_records(path: Path) -> Iterator[dict]:
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"invalid JSON at line {line_number} in {path}"
                ) from error
            if not isinstance(record, dict):
                raise ValueError(f"record at line {line_number} must be an object")
            for field in ("id", "text"):
                if not isinstance(record.get(field), str) or not record[field]:
                    raise ValueError(
                        f"record field {field!r} at line {line_number} must be non-empty"
                    )
            yield record


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def pack_jsonl_gz_to_memmap(
    input_path: Path | str,
    output_path: Path | str,
    tokenizer: object,
    sequence_length: int,
    sequence_count: int,
) -> dict:
    input_path = Path(input_path).resolve()
    output_path = Path(output_path).resolve()
    if not input_path.is_file():
        raise FileNotFoundError(input_path)
    if output_path.exists():
        raise FileExistsError(output_path)
    if sequence_length < 2:
        raise ValueError("sequence_length must be at least 2")
    if sequence_count < 1:
        raise ValueError("sequence_count must be positive")
    eos_token_id = getattr(tokenizer, "eos_token_id", None)
    if eos_token_id is None:
        raise ValueError("tokenizer must define eos_token_id")

    required_tokens = sequence_length * sequence_count
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.", dir=output_path.parent
    )
    os.close(descriptor)
    temporary_path = Path(temporary_name)
    tokens_written = 0
    documents_read = 0
    last_document_id = None
    last_document_tokens_used = 0
    last_document_complete = False

    try:
        packed = np.memmap(
            temporary_path, mode="w+", dtype=np.uint32, shape=(required_tokens,)
        )
        for record in _iter_records(input_path):
            token_ids = tokenizer.encode(record["text"], add_special_tokens=False)
            token_ids.append(eos_token_id)
            if token_ids and (
                min(token_ids) < 0 or max(token_ids) > np.iinfo(np.uint32).max
            ):
                raise ValueError(
                    f"token id outside uint32 range in document {record['id']}"
                )

            documents_read += 1
            last_document_id = record["id"]
            remaining = required_tokens - tokens_written
            last_document_tokens_used = min(remaining, len(token_ids))
            end = tokens_written + last_document_tokens_used
            packed[tokens_written:end] = token_ids[:last_document_tokens_used]
            tokens_written = end
            last_document_complete = last_document_tokens_used == len(token_ids)
            if tokens_written == required_tokens:
                break

        packed.flush()
        del packed
        if tokens_written != required_tokens:
            raise ValueError(
                f"packing requires {required_tokens} tokens, found {tokens_written}"
            )
        os.replace(temporary_path, output_path)
    except BaseException:
        temporary_path.unlink(missing_ok=True)
        raise

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "dtype": "uint32",
        "sequence_length": sequence_length,
        "sequence_count": sequence_count,
        "input_tokens": required_tokens,
        "documents_read": documents_read,
        "last_document_id": last_document_id,
        "last_document_tokens_used": last_document_tokens_used,
        "last_document_complete": last_document_complete,
        "tokenizer": getattr(tokenizer, "name_or_path", type(tokenizer).__name__),
        "sha256": _sha256(output_path),
        "bytes": output_path.stat().st_size,
    }


class TokenMemmapDataset:
    def __init__(
        self, path: Path | str, sequence_length: int, sequence_count: int
    ) -> None:
        self.path = Path(path).resolve()
        self.sequence_length = sequence_length
        self.sequence_count = sequence_count
        expected_bytes = sequence_length * sequence_count * np.dtype(np.uint32).itemsize
        if self.path.stat().st_size != expected_bytes:
            raise ValueError(
                f"memmap size is {self.path.stat().st_size} bytes, expected {expected_bytes}"
            )
        self._tokens = np.memmap(
            self.path,
            mode="r",
            dtype=np.uint32,
            shape=(sequence_count, sequence_length),
        )

    def __len__(self) -> int:
        return self.sequence_count

    def __getitem__(self, index: int) -> dict:
        if index < 0:
            index += self.sequence_count
        if index < 0 or index >= self.sequence_count:
            raise IndexError(index)
        return {"input_ids": self._tokens[index].astype(np.int64)}


In [ ]:
from __future__ import annotations

import gzip
import io
import json
import math
import time
import urllib.request
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator

SUPPORTED_SOURCES = {"s2orc", "s2ag"}


def source_family(source: str) -> str:
    family = source.split("/", maxsplit=1)[0]
    if family not in SUPPORTED_SOURCES:
        raise ValueError(f"unsupported source {source}")
    return family


def validate_record(record: dict, expected_source: str | None = None) -> dict:
    if not isinstance(record, dict):
        raise ValueError("each record must be a JSON object")
    for key in ("id", "source", "text"):
        if not isinstance(record.get(key), str) or not record[key]:
            raise ValueError(f"record field {key!r} must be a non-empty string")
    record_source = source_family(record["source"])
    if expected_source is not None and record_source != source_family(expected_source):
        raise ValueError(
            f"expected source {expected_source}, received {record['source']}"
        )
    return record


def iter_jsonl_gz(url: str) -> Iterator[dict]:
    with urllib.request.urlopen(url, timeout=300) as response:
        with gzip.GzipFile(fileobj=response) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8") as text_stream:
                for line_number, line in enumerate(text_stream, start=1):
                    if not line.strip():
                        continue
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError as error:
                        raise ValueError(
                            f"invalid JSON at line {line_number} from {url}"
                        ) from error
                    yield validate_record(record)


def collect_source_records(
    urls: Iterable[str], limits: dict[str, int | None]
) -> dict[str, list[dict]]:
    unknown_sources = set(limits) - SUPPORTED_SOURCES
    if unknown_sources:
        raise ValueError(f"unsupported requested sources: {sorted(unknown_sources)}")
    for source, limit in limits.items():
        if limit is not None and limit < 1:
            raise ValueError(f"limit for {source} must be positive or None")

    records = {source: [] for source in limits}

    def all_finite_limits_reached() -> bool:
        return all(
            limit is not None and len(records[source]) >= limit
            for source, limit in limits.items()
        )

    for url in urls:
        for record in iter_jsonl_gz(url):
            source = source_family(record["source"])
            if source not in records:
                continue
            limit = limits[source]
            if limit is None or len(records[source]) < limit:
                records[source].append(record)
            if all_finite_limits_reached():
                return records

    short = {
        source: {"expected": limit, "actual": len(records[source])}
        for source, limit in limits.items()
        if limit is not None and len(records[source]) < limit
    }
    if short:
        raise ValueError(f"validation streams ended before limits were met: {short}")
    return records


def iter_packed_sequences(
    records: Iterable[dict], tokenizer: Any, sequence_length: int
) -> Iterator[dict]:
    if sequence_length < 2:
        raise ValueError("sequence_length must be at least 2")
    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None:
        raise ValueError("tokenizer must define eos_token_id")

    token_buffer: list[int] = []
    origin_buffer: list[str] = []

    for raw_record in records:
        record = validate_record(raw_record)
        document_id = record["id"]
        token_ids = tokenizer.encode(record["text"], add_special_tokens=False)
        token_ids.append(eos_token_id)

        position = 0
        while position < len(token_ids):
            space = sequence_length - len(token_buffer)
            next_position = min(position + space, len(token_ids))
            piece = token_ids[position:next_position]
            token_buffer.extend(piece)
            origin_buffer.extend([document_id] * len(piece))
            position = next_position

            if len(token_buffer) == sequence_length:
                yield {
                    "input_ids": token_buffer,
                    "labels": token_buffer.copy(),
                    "document_ids": list(dict.fromkeys(origin_buffer)),
                }
                token_buffer = []
                origin_buffer = []

    if token_buffer:
        padding = sequence_length - len(token_buffer)
        yield {
            "input_ids": token_buffer + [eos_token_id] * padding,
            "labels": token_buffer + [-100] * padding,
            "document_ids": list(dict.fromkeys(origin_buffer)),
        }


def perplexity_from_loss(mean_loss: float) -> float:
    if not math.isfinite(mean_loss):
        raise ValueError("mean loss must be finite")
    try:
        return math.exp(mean_loss)
    except OverflowError:
        return float("inf")


def combine_source_metrics(metrics: Iterable[dict]) -> dict:
    metrics = list(metrics)
    total_tokens = sum(int(item["predicted_tokens"]) for item in metrics)
    if total_tokens <= 0:
        raise ValueError("predicted token total must be positive")
    total_nll = sum(float(item["negative_log_likelihood"]) for item in metrics)
    loss = total_nll / total_tokens
    return {
        "negative_log_likelihood": total_nll,
        "predicted_tokens": total_tokens,
        "loss": loss,
        "perplexity": perplexity_from_loss(loss),
    }


def count_shifted_targets(labels: Any) -> int:
    return int(labels[:, 1:].ne(-100).sum().item())


def _iter_batches(items: Iterable[dict], batch_size: int) -> Iterator[list[dict]]:
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    batch: list[dict] = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def evaluate_source(
    model: Any,
    packed_sequences: Iterable[dict],
    source: str,
    batch_size: int,
    device: str,
    log_every_steps: int,
    progress_callback: Callable[[dict], None] | None = None,
) -> dict:
    import torch

    if source not in SUPPORTED_SOURCES:
        raise ValueError(f"unsupported source {source}")
    if log_every_steps < 1:
        raise ValueError("log_every_steps must be positive")

    start = time.perf_counter()
    total_nll = 0.0
    predicted_tokens = 0
    input_tokens = 0
    sequence_count = 0
    batch_count = 0
    document_ids: list[str] = []
    seen_document_ids: set[str] = set()
    last_logged_batch = 0

    try:
        with torch.inference_mode():
            for batch_count, batch in enumerate(
                _iter_batches(packed_sequences, batch_size), start=1
            ):
                input_ids = torch.tensor(
                    [item["input_ids"] for item in batch],
                    dtype=torch.long,
                    device=device,
                )
                labels = torch.tensor(
                    [item["labels"] for item in batch],
                    dtype=torch.long,
                    device=device,
                )
                attention_mask = labels.ne(-100).long()
                batch_targets = count_shifted_targets(labels)
                if batch_targets < 1:
                    raise ValueError("a packed batch contained no prediction targets")

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                batch_loss = float(outputs.loss.item())
                if not math.isfinite(batch_loss):
                    raise ValueError(
                        f"non-finite loss for {source} at batch {batch_count}"
                    )

                total_nll += batch_loss * batch_targets
                predicted_tokens += batch_targets
                input_tokens += int(labels.ne(-100).sum().item())
                sequence_count += len(batch)
                for item in batch:
                    for document_id in item["document_ids"]:
                        if document_id not in seen_document_ids:
                            seen_document_ids.add(document_id)
                            document_ids.append(document_id)

                if progress_callback is not None and (
                    batch_count == 1 or batch_count % log_every_steps == 0
                ):
                    elapsed = max(time.perf_counter() - start, 1e-9)
                    running_loss = total_nll / predicted_tokens
                    progress_callback(
                        {
                            "source": source,
                            "batch": batch_count,
                            "sequences": sequence_count,
                            "predicted_tokens": predicted_tokens,
                            "loss": running_loss,
                            "perplexity": perplexity_from_loss(running_loss),
                            "tokens_per_second": predicted_tokens / elapsed,
                            "elapsed_seconds": elapsed,
                            "gpu_memory_gb": torch.cuda.memory_allocated() / (1024**3),
                        }
                    )
                    last_logged_batch = batch_count
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f"CUDA ran out of memory with batch_size={batch_size}; "
            "reduce CONFIG['batch_size'] and rerun"
        ) from error

    if predicted_tokens < 1:
        raise ValueError(f"no prediction targets were evaluated for {source}")

    elapsed = max(time.perf_counter() - start, 1e-9)
    loss = total_nll / predicted_tokens
    result = {
        "source": source,
        "documents": len(document_ids),
        "document_ids": document_ids,
        "sequences": sequence_count,
        "batches": batch_count,
        "input_tokens": input_tokens,
        "predicted_tokens": predicted_tokens,
        "negative_log_likelihood": total_nll,
        "loss": loss,
        "perplexity": perplexity_from_loss(loss),
        "elapsed_seconds": elapsed,
        "tokens_per_second": predicted_tokens / elapsed,
    }

    if progress_callback is not None and last_logged_batch != batch_count:
        progress_callback({**result, "batch": batch_count})
    return result


def write_result_json(path: str | Path, result: dict) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(
            result, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False
        )
        + "\n",
        encoding="utf-8",
    )
    return output_path


## 1. Select one variant and validate the runtime

Change only `VARIANT`. The notebook stops before downloading data if the Python, package, or GPU requirements are wrong.


In [ ]:
import gc
import importlib.metadata
import json
import math
import os
import platform
import random
import sys
from pathlib import Path

import boto3
import causal_conv1d
import mamba_ssm
import numpy as np
import torch
import transformers
import wandb
from google.colab import userdata
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments


VARIANT = "raw"
ALLOWED_VARIANTS = {"raw", "minhashlsh", "lshbloom"}

CONFIG = {
    "model_id": "pfnet/plamo-2-1b",
    "model_revision": "92c75fd6eea9018bcb9c33ee8921589febe071fa",
    "required_gpu_substring": "A100",
    "torch_version": "2.5.1",
    "transformers_version": "4.57.1",
    "mamba_ssm_version": "2.2.4",
    "causal_conv1d_version": "1.4.0",
    "sequence_length": 2048,
    "sequence_count": 12_150,
    "train_input_tokens": 24_883_200,
    "available_plamo_tokens": {
        "raw": 27_161_292,
        "minhashlsh": 24_930_000,
        "lshbloom": 24_883_479,
    },
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-5,
    "warmup_ratio": 0.03,
    "weight_decay": 0.1,
    "max_grad_norm": 1.0,
    "compute_dtype": "bfloat16",
    "fp16": False,
    "bf16": True,
    "seed": 42,
    "save_steps": 250,
    "logging_steps": 10,
    "eval_batch_size": 1,
    "s2orc_documents": 320,
    "s2ag_documents": 680,
    "wandb_project": "lshbloom-pes2o",
    "wandb_group": "plamo-2-1b-pes2o-dedup-equal-24883200",
}

if VARIANT not in ALLOWED_VARIANTS:
    raise ValueError(f"VARIANT must be one of {sorted(ALLOWED_VARIANTS)}")
if CONFIG["sequence_length"] * CONFIG["sequence_count"] != CONFIG["train_input_tokens"]:
    raise RuntimeError("Training token budget is internally inconsistent")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("PLaMo requires the Colab 2025.07 Python 3.11 runtime")
if Version(torch.__version__.split("+")[0]) != Version(CONFIG["torch_version"]):
    raise RuntimeError(f"Expected torch {CONFIG['torch_version']}, received {torch.__version__}")
if Version(transformers.__version__) != Version(CONFIG["transformers_version"]):
    raise RuntimeError(
        f"Expected transformers {CONFIG['transformers_version']}, received {transformers.__version__}"
    )
for distribution, expected in (
    ("mamba-ssm", CONFIG["mamba_ssm_version"]),
    ("causal-conv1d", CONFIG["causal_conv1d_version"]),
):
    actual = importlib.metadata.version(distribution)
    if Version(actual) != Version(expected):
        raise RuntimeError(f"Expected {distribution} {expected}, received {actual}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime in Colab.")
gpu_name = torch.cuda.get_device_name(0)
if CONFIG["required_gpu_substring"] not in gpu_name:
    raise RuntimeError(f"This experiment requires an A100; received {gpu_name}")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError(f"BF16 is unavailable on {gpu_name}")

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])


def optional_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


try:
    AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
    AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
    WANDB_API_KEY = userdata.get("WANDB_API_KEY")
except Exception as error:
    raise RuntimeError("One or more required Colab secrets are unavailable") from error

for name, value in (
    ("AWS_ACCESS_KEY_ID", AWS_ACCESS_KEY_ID),
    ("AWS_SECRET_ACCESS_KEY", AWS_SECRET_ACCESS_KEY),
    ("WANDB_API_KEY", WANDB_API_KEY),
):
    if not value:
        raise RuntimeError(f"Missing required Colab secret: {name}")

session_kwargs = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region_name": optional_secret("AWS_DEFAULT_REGION") or "ap-northeast-1",
}
AWS_SESSION_TOKEN = optional_secret("AWS_SESSION_TOKEN")
if AWS_SESSION_TOKEN:
    session_kwargs["aws_session_token"] = AWS_SESSION_TOKEN
s3 = boto3.session.Session(**session_kwargs).client("s3")

S3_BUCKET = "calista-bucket"
S3_PREFIX = "pes2o/v2/experiments/pilot-5000/"
S3_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}"
checkpoint_prefix = f"{S3_PREFIX}plamo-2-1b-equal-24883200/checkpoints/{VARIANT}"
WORK_DIR = Path(f"/content/plamo-2-1b-pes2o-{VARIANT}-equal-24883200")
DATA_PATH = WORK_DIR / "train.jsonl.gz"
MANIFEST_PATH = WORK_DIR / "manifest.json"
MEMMAP_PATH = WORK_DIR / "train-tokens.uint32"
TRAINER_DIR = WORK_DIR / "trainer"
FINAL_MODEL_DIR = WORK_DIR / "final"
RESULT_PATH = WORK_DIR / "result.json"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", gpu_name)
print("Variant:", VARIANT)
print("PLaMo training tokens:", f"{CONFIG['train_input_tokens']:,}")
print("S3 output:", f"s3://{S3_BUCKET}/{checkpoint_prefix}/")

# Fail before training if this runtime cannot persist its final output.
preflight_key = f"{checkpoint_prefix}/_write-preflight.txt"
try:
    s3.put_object(Bucket=S3_BUCKET, Key=preflight_key, Body=b"ok")
finally:
    s3.delete_object(Bucket=S3_BUCKET, Key=preflight_key)
print("S3 read/write preflight passed.")


## 2. Download the selected dataset and pack exactly 25 million PLaMo tokens

The source manifest token counts were produced with the earlier Qwen tokenizer, so they are recorded only as source metadata. This notebook does not use them to approve the PLaMo budget. The PLaMo tokenizer reads documents in their existing order, inserts EOS between documents, and stops after exactly 12,207 complete 2,048-token sequences. Packing fails if the selected variant does not contain enough PLaMo tokens.


In [ ]:
s3.download_file(S3_BUCKET, f"{S3_PREFIX}manifest.json", str(MANIFEST_PATH))
source_manifest_sha256 = _sha256(MANIFEST_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
variant_info = manifest["variants"][VARIANT]

s3.download_file(
    S3_BUCKET,
    f"{S3_PREFIX}{variant_info['relative_path']}",
    str(DATA_PATH),
)
actual_sha256 = _sha256(DATA_PATH)
if actual_sha256 != variant_info["sha256"]:
    raise RuntimeError("Dataset SHA-256 mismatch")
available_tokens = CONFIG["available_plamo_tokens"][VARIANT]
if available_tokens < CONFIG["train_input_tokens"]:
    raise RuntimeError(
        f"{VARIANT} has {available_tokens:,} PLaMo tokens, but the experiment "
        f"requires {CONFIG['train_input_tokens']:,}"
    )

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    trust_remote_code=True,
)
if tokenizer.eos_token_id is None:
    raise RuntimeError("PLaMo tokenizer has no EOS token")

packing = pack_jsonl_gz_to_memmap(
    input_path=DATA_PATH,
    output_path=MEMMAP_PATH,
    tokenizer=tokenizer,
    sequence_length=CONFIG["sequence_length"],
    sequence_count=CONFIG["sequence_count"],
)
train_dataset = TokenMemmapDataset(
    MEMMAP_PATH,
    sequence_length=CONFIG["sequence_length"],
    sequence_count=CONFIG["sequence_count"],
)


def causal_lm_collator(features):
    input_ids = torch.from_numpy(
        np.stack([feature["input_ids"] for feature in features])
    ).long()
    return {"input_ids": input_ids, "labels": input_ids.clone()}


print(json.dumps(packing, indent=2))
print("Optimizer updates:", math.ceil(len(train_dataset) / CONFIG["gradient_accumulation_steps"]))


## 3. Start W&B and run a full forward/backward smoke test

PLaMo is loaded with its pinned remote model code. Parameters remain FP32 while A100 computation uses BF16 autocast. BF16 has the exponent range needed to avoid the FP16 overflow observed with this model. The smoke test uses a complete 2,048-token sequence and checks both the loss and gradient norm before the long training starts.


In [ ]:
wandb.login(key=WANDB_API_KEY)
run = wandb.init(
    project=CONFIG["wandb_project"],
    group=CONFIG["wandb_group"],
    name=f"plamo-2-1b-{VARIANT}-equal-24883200",
    config={
        **CONFIG,
        "variant": VARIANT,
        "source_manifest_sha256": source_manifest_sha256,
        "dataset_sha256": actual_sha256,
        "source_manifest_token_count": variant_info["token_count"],
        "available_plamo_tokens": available_tokens,
        "unused_tail_tokens": available_tokens - CONFIG["train_input_tokens"],
        "packing": packing,
        "gpu": gpu_name,
    },
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    trust_remote_code=True,
    torch_dtype=torch.float32,
).to("cuda")
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.train()

smoke_batch = {
    key: value.to("cuda")
    for key, value in causal_lm_collator([train_dataset[0]]).items()
}
torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.bfloat16):
    smoke_output = model(**smoke_batch)
if not torch.isfinite(smoke_output.loss):
    raise RuntimeError(f"Smoke test loss is not finite: {smoke_output.loss.item()}")
smoke_output.loss.backward()
smoke_grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=float("inf")
)
if not torch.isfinite(smoke_grad_norm):
    raise RuntimeError(f"Smoke test gradient norm is not finite: {smoke_grad_norm.item()}")
smoke_peak_bytes = torch.cuda.max_memory_allocated()
model.zero_grad(set_to_none=True)
wandb.log({
    "smoke_test/passed": 1,
    "smoke_test/loss": smoke_output.loss.item(),
    "smoke_test/gradient_norm": smoke_grad_norm.item(),
    "smoke_test/peak_cuda_bytes": smoke_peak_bytes,
})
print(
    f"Smoke test passed: loss={smoke_output.loss.item():.4f}, "
    f"gradient_norm={smoke_grad_norm.item():.4f}, "
    f"peak GPU memory={smoke_peak_bytes / (1024**3):.2f} GiB"
)
del smoke_batch, smoke_output, smoke_grad_norm
gc.collect()
torch.cuda.empty_cache()


## 4. Continue pretraining and persist the final checkpoint

All three variants use the same 24,883,200-token budget and schedule. Adafactor is used for all three PLaMo runs, and BF16 computation is used with FP32 model parameters for numerical stability. The final model is uploaded to a new S3 experiment prefix immediately after training, before validation begins.


In [ ]:
training_args = TrainingArguments(
    output_dir=str(TRAINER_DIR),
    overwrite_output_dir=True,
    num_train_epochs=1.0,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    fp16=CONFIG["fp16"],
    bf16=CONFIG["bf16"],
    gradient_checkpointing=True,
    logging_steps=CONFIG["logging_steps"],
    logging_first_step=True,
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=1,
    save_only_model=True,
    report_to=["wandb"],
    run_name=f"plamo-2-1b-{VARIANT}-equal-24883200",
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
    dataloader_num_workers=2,
    remove_unused_columns=False,
    optim="adafactor",
    save_safetensors=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=causal_lm_collator,
    processing_class=tokenizer,
)
train_output = trainer.train()
train_metrics = dict(train_output.metrics)
train_metrics["train_input_tokens"] = CONFIG["train_input_tokens"]
train_metrics["variant"] = VARIANT
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(FINAL_MODEL_DIR)

for local_path in FINAL_MODEL_DIR.rglob("*"):
    if local_path.is_file():
        relative = local_path.relative_to(FINAL_MODEL_DIR).as_posix()
        s3.upload_file(
            str(local_path),
            S3_BUCKET,
            f"{checkpoint_prefix}/final/{relative}",
        )
s3.upload_file(
    str(TRAINER_DIR / "train_results.json"),
    S3_BUCKET,
    f"{checkpoint_prefix}/train_results.json",
)
run.summary["checkpoint_s3_uri"] = f"s3://{S3_BUCKET}/{checkpoint_prefix}/final/"
print("Training complete:", json.dumps(train_metrics, indent=2))
print("Checkpoint safely stored at:", run.summary["checkpoint_s3_uri"])
del trainer
gc.collect()
torch.cuda.empty_cache()


## 5. Evaluate the same 1,000-document peS2o validation sample

The fixed sample contains 320 S2ORC documents and 680 S2AG documents. The validation files are pinned to an immutable peS2o revision. Evaluation uses BF16 autocast and batch size 1 on the A100.


In [ ]:
VALIDATION_REVISION = "636a503e44a3ca1b58e01fb61eab0825cd574de0"
VALIDATION_URLS = [
    f"https://huggingface.co/datasets/allenai/peS2o/resolve/{VALIDATION_REVISION}/data/v2/validation-00000-of-00002.json.gz",
    f"https://huggingface.co/datasets/allenai/peS2o/resolve/{VALIDATION_REVISION}/data/v2/validation-00001-of-00002.json.gz",
]
source_limits = {
    "s2orc": CONFIG["s2orc_documents"],
    "s2ag": CONFIG["s2ag_documents"],
}
records_by_source = collect_source_records(VALIDATION_URLS, source_limits)


def log_eval_progress(metrics):
    prefix = f"eval/{metrics['source']}"
    wandb.log({
        f"{prefix}/batch": metrics["batch"],
        f"{prefix}/running_loss": metrics["loss"],
        f"{prefix}/running_perplexity": metrics["perplexity"],
        f"{prefix}/predicted_tokens": metrics["predicted_tokens"],
        f"{prefix}/tokens_per_second": metrics["tokens_per_second"],
    })


model.eval()
source_results = {}
for source in ("s2orc", "s2ag"):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        source_results[source] = evaluate_source(
            model=model,
            packed_sequences=iter_packed_sequences(
                records_by_source[source], tokenizer, CONFIG["sequence_length"]
            ),
            source=source,
            batch_size=CONFIG["eval_batch_size"],
            device="cuda",
            log_every_steps=25,
            progress_callback=log_eval_progress,
        )
    print(
        f"{source}: loss={source_results[source]['loss']:.4f}, "
        f"PPL={source_results[source]['perplexity']:.4f}"
    )

overall = combine_source_metrics(source_results.values())
wandb.log({
    "eval/overall_loss": overall["loss"],
    "eval/overall_perplexity": overall["perplexity"],
    "eval/overall_predicted_tokens": overall["predicted_tokens"],
    "eval/s2orc_perplexity": source_results["s2orc"]["perplexity"],
    "eval/s2ag_perplexity": source_results["s2ag"]["perplexity"],
})
run.summary["eval/overall_perplexity"] = overall["perplexity"]
run.summary["eval/s2orc_perplexity"] = source_results["s2orc"]["perplexity"]
run.summary["eval/s2ag_perplexity"] = source_results["s2ag"]["perplexity"]
print(f"Overall: loss={overall['loss']:.4f}, PPL={overall['perplexity']:.4f}")


## 6. Save the reproducibility record to S3 and W&B

The result JSON records the pinned model and validation revisions, exact PLaMo packing hash, source file hash, software versions, training metrics, and validation metrics.


In [ ]:
result = {
    "schema_version": 1,
    "variant": VARIANT,
    "config": CONFIG,
    "model": {
        "id": CONFIG["model_id"],
        "revision": CONFIG["model_revision"],
        "trust_remote_code": True,
    },
    "dataset": {
        "s3_uri": f"{S3_URI}{variant_info['relative_path']}",
        "source_file_sha256": actual_sha256,
        "source_manifest_sha256": source_manifest_sha256,
        "source_manifest_token_count_qwen_tokenizer": variant_info["token_count"],
        "full_file_plamo_token_count": available_tokens,
        "unused_tail_tokens": available_tokens - CONFIG["train_input_tokens"],
        "plamo_packing": packing,
    },
    "training": train_metrics,
    "validation": {
        "dataset_revision": VALIDATION_REVISION,
        "urls": VALIDATION_URLS,
        "sources": source_results,
        "overall": overall,
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "mamba_ssm": importlib.metadata.version("mamba-ssm"),
        "causal_conv1d": importlib.metadata.version("causal-conv1d"),
        "wandb": wandb.__version__,
        "gpu": gpu_name,
    },
    "wandb_run_id": run.id,
    "wandb_run_url": run.url,
}
RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
s3.upload_file(str(RESULT_PATH), S3_BUCKET, f"{checkpoint_prefix}/result.json")

artifact = wandb.Artifact(
    name=f"plamo-2-1b-{VARIANT}-equal-24883200-results",
    type="evaluation",
    metadata={
        "variant": VARIANT,
        "train_input_tokens": CONFIG["train_input_tokens"],
        "model_revision": CONFIG["model_revision"],
    },
)
artifact.add_file(str(RESULT_PATH))
run.log_artifact(artifact)
print("W&B:", run.url)
print("Checkpoint:", run.summary["checkpoint_s3_uri"])
print("Result:", f"s3://{S3_BUCKET}/{checkpoint_prefix}/result.json")
wandb.finish()


## Reading the three runs

Compare `eval/overall_perplexity`, `eval/s2orc_perplexity`, `eval/s2ag_perplexity`, and `train/train_loss` inside the W&B group `plamo-2-1b-pes2o-dedup-equal-24883200`. Lower perplexity is better. Because Raw, MinHashLSH, and LSHBloom each use the same 24,883,200-token PLaMo budget, their final differences estimate the effect of which content remains after deduplication. Do not mix these runs with checkpoints from the earlier 24,999,936-token prefix. They do not measure token-cost savings; the full-corpus efficiency experiment serves that separate question.
